# Set up for dataset and model

Package installation, loading, and dataloaders. There's also a resnet18 model defined.

In [1]:
# !pip install tensorboardX

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import time
import matplotlib.pyplot as plt
from tqdm import tqdm

from torchvision import datasets, transforms
# from tensorboardX import SummaryWriter

use_cuda = True
device = torch.device("cuda" if use_cuda else "cpu")
batch_size = 64

np.random.seed(42)
torch.manual_seed(42)


## Dataloaders
train_dataset = datasets.CIFAR10('cifar10_data/', train=True, download=True, transform=transforms.Compose(
    [transforms.ToTensor()]
))
test_dataset = datasets.CIFAR10('cifar10_data/', train=False, download=True, transform=transforms.Compose(
    [transforms.ToTensor()]
))

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False)



In [2]:

def tp_relu(x, delta=1.):
    ind1 = (x < -1. * delta).float()
    ind2 = (x > delta).float()
    return .5 * (x + delta) * (1 - ind1) * (1 - ind2) + x * ind2

def tp_smoothed_relu(x, delta=1.):
    ind1 = (x < -1. * delta).float()
    ind2 = (x > delta).float()
    return (x + delta) ** 2 / (4 * delta) * (1 - ind1) * (1 - ind2) + x * ind2

class Normalize(nn.Module):
    def __init__(self, mu, std):
        super(Normalize, self).__init__()
        self.mu, self.std = mu, std

    def forward(self, x):
        return (x - self.mu) / self.std

class IdentityLayer(nn.Module):
    def forward(self, inputs):
        return inputs
    
class PreActBlock(nn.Module):
    '''Pre-activation version of the BasicBlock.'''
    expansion = 1

    def __init__(self, in_planes, planes, bn, learnable_bn, stride=1, activation='relu'):
        super(PreActBlock, self).__init__()
        self.collect_preact = True
        self.activation = activation
        self.avg_preacts = []
        self.bn1 = nn.BatchNorm2d(in_planes, affine=learnable_bn) if bn else IdentityLayer()
        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=not learnable_bn)
        self.bn2 = nn.BatchNorm2d(planes, affine=learnable_bn) if bn else IdentityLayer()
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=1, padding=1, bias=not learnable_bn)

        if stride != 1 or in_planes != self.expansion*planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, self.expansion*planes, kernel_size=1, stride=stride, bias=not learnable_bn)
            )

    def act_function(self, preact):
        if self.activation == 'relu':
            act = F.relu(preact)
        elif self.activation[:6] == '3prelu':
            act = tp_relu(preact, delta=float(self.activation.split('relu')[1]))
        elif self.activation[:8] == '3psmooth':
            act = tp_smoothed_relu(preact, delta=float(self.activation.split('smooth')[1]))
        else:
            assert self.activation[:8] == 'softplus'
            beta = int(self.activation.split('softplus')[1])
            act = F.softplus(preact, beta=beta)
        return act

    def forward(self, x):
        out = self.act_function(self.bn1(x))
        shortcut = self.shortcut(out) if hasattr(self, 'shortcut') else x  # Important: using out instead of x
        out = self.conv1(out)
        out = self.conv2(self.act_function(self.bn2(out)))
        out += shortcut
        return out

class PreActResNet(nn.Module):
    def __init__(self, block, num_blocks, n_cls, cuda=True, half_prec=False,
        activation='relu', fts_before_bn=False, normal='none'):
        super(PreActResNet, self).__init__()
        self.bn = True
        self.learnable_bn = True  # doesn't matter if self.bn=False
        self.in_planes = 64
        self.avg_preact = None
        self.activation = activation
        self.fts_before_bn = fts_before_bn
        if normal == 'cifar10':
            self.mu = torch.tensor((0.4914, 0.4822, 0.4465)).view(1, 3, 1, 1)
            self.std = torch.tensor((0.2471, 0.2435, 0.2616)).view(1, 3, 1, 1)
        else:
            self.mu = torch.tensor((0.0, 0.0, 0.0)).view(1, 3, 1, 1)
            self.std = torch.tensor((1.0, 1.0, 1.0)).view(1, 3, 1, 1)
            print('no input normalization')
        if cuda:
            self.mu = self.mu.cuda()
            self.std = self.std.cuda()
        if half_prec:
            self.mu = self.mu.half()
            self.std = self.std.half()

        self.normalize = Normalize(self.mu, self.std)
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=not self.learnable_bn)
        self.layer1 = self._make_layer(block, 64, num_blocks[0], stride=1)
        self.layer2 = self._make_layer(block, 128, num_blocks[1], stride=2)
        self.layer3 = self._make_layer(block, 256, num_blocks[2], stride=2)
        self.layer4 = self._make_layer(block, 512, num_blocks[3], stride=2)
        self.bn = nn.BatchNorm2d(512 * block.expansion)
        self.linear = nn.Linear(512*block.expansion, n_cls)

    def _make_layer(self, block, planes, num_blocks, stride):
        strides = [stride] + [1]*(num_blocks-1)
        layers = []
        for stride in strides:
            layers.append(block(self.in_planes, planes, self.bn, self.learnable_bn, stride, self.activation))
            # layers.append(block(self.in_planes, planes, stride))
            self.in_planes = planes * block.expansion
        return nn.Sequential(*layers)

    def forward(self, x, return_features=False):
        for layer in [*self.layer1, *self.layer2, *self.layer3, *self.layer4]:
            layer.avg_preacts = []

        out = self.normalize(x)
        out = self.conv1(out)
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        if return_features and self.fts_before_bn:
            return out.view(out.size(0), -1)
        out = F.relu(self.bn(out))
        if return_features:
            return out.view(out.size(0), -1)
        out = F.avg_pool2d(out, 4)
        out = out.view(out.size(0), -1)
        out = self.linear(out)

        return out


def PreActResNet18(n_cls, cuda=True, half_prec=False, activation='relu', fts_before_bn=False,
    normal='none'):
    #print('initializing PA RN-18 with act {}, normal {}'.format())
    return PreActResNet(PreActBlock, [2, 2, 2, 2], n_cls=n_cls, cuda=cuda, half_prec=half_prec,
        activation=activation, fts_before_bn=fts_before_bn, normal=normal)


# intialize the model
model = PreActResNet18(10, cuda=True, activation='softplus1').to(device)
model.eval()

no input normalization


PreActResNet(
  (normalize): Normalize()
  (conv1): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (layer1): Sequential(
    (0): PreActBlock(
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    )
    (1): PreActBlock(
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    )
  )
  (layer2): Sequential(
    (0): PreActBloc

# Implement the Attacks

Functions are given a simple useful signature that you can start with. Feel free to extend the signature as you see fit.

You may find it useful to create a 'batched' version of PGD that you can use to create the adversarial attack.

In [3]:
def pgd_linf_untargeted(model, x, labels, k, eps, eps_step):
    model.eval()
    ce_loss = torch.nn.CrossEntropyLoss()
    adv_x = x.clone().detach()
    adv_x.requires_grad_(True) 
    for _ in range(k):
        adv_x.requires_grad_(True)
        model.zero_grad()
        output = model(adv_x)
        # Calculate the loss for untargeted attack (maximize loss for true labels)
        loss = ce_loss(output, labels)
        loss.backward()
        # Compute the adversarial example using FGSM step
        with torch.no_grad():
            # Get the sign of the gradient
            grad_sign = adv_x.grad.sign()
            # Take a step in the direction that increases loss (untargeted attack)
            adv_x = adv_x + eps_step * grad_sign
            # Project back onto the L-infinity ball around original input
            delta = torch.clamp(adv_x - x, -eps, eps)
            # Clamp to valid image range [0, 1]
            adv_x = torch.clamp(x + delta, 0, 1)
   
    return adv_x

In [4]:
def pgd_l2_untargeted(model, x, labels, k, eps, eps_step):
    model.eval()
    ce_loss = torch.nn.CrossEntropyLoss()
    adv_x = x.clone().detach()
    adv_x.requires_grad_(True) 
    for _ in range(k):
        adv_x.requires_grad_(True)
        model.zero_grad()
        output = model(adv_x)
        batch_size = x.size()[0]
        # Calculate the loss for untargeted attack (maximize loss for true labels)
        loss = ce_loss(output, labels)
        loss.backward()
        grad = adv_x.grad.data
        # Compute the adversarial example using L2 PGD step
        with torch.no_grad():
            # Flatten the gradient for L2 norm computation
            grad_flat = grad.view(batch_size, -1)
            # Compute L2 norm of gradient for each sample and avoid division by zero
            grad_norm = torch.clamp(torch.norm(grad_flat, p=2, dim=1, keepdim=True), min=1e-12)
            # Normalize gradient to unit L2 norm
            grad_normalized = grad_flat / grad_norm
            # Reshape back to original shape
            grad_normalized = grad_normalized.view_as(grad)
            # Take a step in the direction that increases loss
            adv_x = adv_x + eps_step * grad_normalized
            # Project back onto the L2 ball around original input
            delta = adv_x - x
            delta_flat = delta.view(batch_size, -1)
            delta_norm = torch.norm(delta_flat, p=2, dim=1, keepdim=True)
            # Clamp the L2 norm of delta to eps
            delta_flat = delta_flat * torch.clamp(eps / delta_norm, max=1.0)
            delta = delta_flat.view_as(x)
            adv_x = x + delta
            # Clamp to valid image range [0, 1]
            adv_x = torch.clamp(adv_x, 0, 1)
            # Reset gradient for next iteration
            adv_x.requires_grad_(True)
   
    return adv_x

# Evaluate Single and Multi-Norm Robust Accuracy

In this section, we evaluate the model on the Linf and L2 attacks as well as union accuracy.

In [5]:
def test_model_on_single_attack(model, attack='pgd_linf', eps=0.1, k=10):
    model.eval()
    tot_test, tot_acc, tot_std_acc = 0.0, 0.0, 0.0
    for batch_idx, (x_batch, y_batch) in tqdm(enumerate(test_loader), total=len(test_loader), desc="Evaluating"):
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)
        if attack == 'pgd_linf':
            # Get adversarial examples using PGD L-infinity attack
            x_adv = pgd_linf_untargeted(model, x_batch, y_batch, k=k, eps=eps, eps_step=eps/4)
        elif attack == 'pgd_l2':
            # Get adversarial examples using PGD L2 attack
            x_adv = pgd_l2_untargeted(model, x_batch, y_batch, k=k, eps=eps, eps_step=eps/4)
        else:
            pass
        
        # Get the testing accuracy and update tot_test and tot_acc
        with torch.no_grad():
            # Compute robust accuracy
            output = model(x_adv)
            pred = torch.max(output, dim=1)[1]
            tot_acc += (pred == y_batch).sum().item()
            
            # Compute standard accuracy
            std_output = model(x_batch)
            std_pred = torch.max(std_output, dim=1)[1]
            tot_std_acc += (std_pred == y_batch).sum().item()
            
            tot_test += y_batch.size(0)
            
    print('Standard accuracy %.5lf' % (tot_std_acc/tot_test))
    print('Robust accuracy %.5lf' % (tot_acc/tot_test), f'on {attack} attack with eps = {eps}')

## Model Training & Evaluation (HW3)

In [6]:
def train_adv_model(model, num_epochs, eps, k, attack):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)

    for epoch in range(num_epochs):
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            if attack == 'pgd_linf':
                adv_images = pgd_linf_untargeted(model, images, labels, k, eps, eps/4)
            elif attack == 'pgd_l2':
                adv_images = pgd_l2_untargeted(model, images, labels, k, eps, eps/4)
            else:
                raise ValueError(f"Unsupported attack: {attack}")
            
            model.train()
            optimizer.zero_grad()
            outputs = model(adv_images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {loss.item():.4f}")

    return model
    

In [7]:
def train_model(model, num_epochs):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)

    for epoch in range(num_epochs):
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {loss.item():.4f}")

    return model


In [8]:
def test_model(model):
    model.eval()
    total_correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = outputs.max(1)
            total_correct += (predicted == labels).sum().item()
            total += labels.size(0)
    return total_correct / total

### Adversarial Training

In [23]:
# Adversarial Training
model = PreActResNet18(10, cuda=True, activation='softplus1').to(device)
adv_model = train_adv_model(model, num_epochs=10, eps=8/255, k=10, attack='pgd_linf')


no input normalization
Epoch 1/10, Loss: 1.8239
Epoch 2/10, Loss: 1.7031
Epoch 3/10, Loss: 1.7810


KeyboardInterrupt: 

In [21]:
## Clean Accuracy
print("Test Accuracy of Adversarial Model: ", test_model(adv_model))

Test Accuracy of Adversarial Model:  0.6108


In [11]:
## Linf Robust Accuracy
for epsilon in [2/255, 4/255, 8/255, 16/255]:
    test_model_on_single_attack(adv_model, attack='pgd_linf', eps=epsilon, k=10)

Evaluating: 100%|██████████| 157/157 [00:13<00:00, 11.35it/s]


Standard accuracy 0.61080
Robust accuracy 0.54190 on pgd_linf attack with eps = 0.00784313725490196


Evaluating: 100%|██████████| 157/157 [00:13<00:00, 11.37it/s]


Standard accuracy 0.61080
Robust accuracy 0.47660 on pgd_linf attack with eps = 0.01568627450980392


Evaluating: 100%|██████████| 157/157 [00:13<00:00, 11.39it/s]


Standard accuracy 0.61080
Robust accuracy 0.34500 on pgd_linf attack with eps = 0.03137254901960784


Evaluating: 100%|██████████| 157/157 [00:13<00:00, 11.35it/s]

Standard accuracy 0.61080
Robust accuracy 0.14850 on pgd_linf attack with eps = 0.06274509803921569


In [12]:
## FGSM Linf Robust Accuracy
for epsilon in [2/255, 4/255, 8/255, 16/255]:
    test_model_on_single_attack(adv_model, attack='pgd_linf', eps=epsilon, k=1)

Evaluating: 100%|██████████| 157/157 [00:02<00:00, 69.21it/s]


Standard accuracy 0.61080
Robust accuracy 0.59620 on pgd_linf attack with eps = 0.00784313725490196


Evaluating: 100%|██████████| 157/157 [00:02<00:00, 69.31it/s]


Standard accuracy 0.61080
Robust accuracy 0.57780 on pgd_linf attack with eps = 0.01568627450980392


Evaluating: 100%|██████████| 157/157 [00:02<00:00, 68.90it/s]


Standard accuracy 0.61080
Robust accuracy 0.54310 on pgd_linf attack with eps = 0.03137254901960784


Evaluating: 100%|██████████| 157/157 [00:02<00:00, 68.49it/s]

Standard accuracy 0.61080
Robust accuracy 0.48010 on pgd_linf attack with eps = 0.06274509803921569


In [13]:
## L2 Robust Accuracy
for epsilon in [0.25, 0.5, 0.75]:
    test_model_on_single_attack(adv_model, attack='pgd_l2', eps=epsilon, k=10)

Evaluating: 100%|██████████| 157/157 [00:13<00:00, 11.23it/s]


Standard accuracy 0.61080
Robust accuracy 0.53820 on pgd_l2 attack with eps = 0.25


Evaluating: 100%|██████████| 157/157 [00:13<00:00, 11.38it/s]


Standard accuracy 0.61080
Robust accuracy 0.46870 on pgd_l2 attack with eps = 0.5


Evaluating: 100%|██████████| 157/157 [00:13<00:00, 11.36it/s]

Standard accuracy 0.61080
Robust accuracy 0.39290 on pgd_l2 attack with eps = 0.75


In [14]:
## FGSM L2 Robust Accuracy
for epsilon in [0.25, 0.5, 0.75]:
    test_model_on_single_attack(adv_model, attack='pgd_l2', eps=epsilon, k=1)

Evaluating: 100%|██████████| 157/157 [00:02<00:00, 69.53it/s]


Standard accuracy 0.61080
Robust accuracy 0.59500 on pgd_l2 attack with eps = 0.25


Evaluating: 100%|██████████| 157/157 [00:02<00:00, 69.09it/s]


Standard accuracy 0.61080
Robust accuracy 0.57720 on pgd_l2 attack with eps = 0.5


Evaluating: 100%|██████████| 157/157 [00:02<00:00, 69.32it/s]

Standard accuracy 0.61080
Robust accuracy 0.55810 on pgd_l2 attack with eps = 0.75


### Normal Training

In [15]:
norm_model = PreActResNet18(10, cuda=True, activation='softplus1').to(device)
normal_model = train_model(norm_model, num_epochs=10)

no input normalization
Epoch 1/10, Loss: 1.9381
Epoch 2/10, Loss: 0.8062
Epoch 3/10, Loss: 0.4532
Epoch 4/10, Loss: 0.9741
Epoch 5/10, Loss: 0.8933
Epoch 6/10, Loss: 0.5052
Epoch 7/10, Loss: 0.2533
Epoch 8/10, Loss: 0.1087
Epoch 9/10, Loss: 0.1577
Epoch 10/10, Loss: 0.0677


In [16]:
## Clean Accuracy
print("Test Accuracy of Normal Model: ", test_model(normal_model))

Test Accuracy of Normal Model:  0.7824


In [17]:
## Linf Robust Accuracy
for epsilon in [2/255, 4/255, 8/255, 16/255]:
    test_model_on_single_attack(normal_model, attack='pgd_linf', eps=epsilon)

Evaluating: 100%|██████████| 157/157 [00:13<00:00, 11.37it/s]


Standard accuracy 0.78240
Robust accuracy 0.07500 on pgd_linf attack with eps = 0.00784313725490196


Evaluating: 100%|██████████| 157/157 [00:13<00:00, 11.36it/s]


Standard accuracy 0.78240
Robust accuracy 0.00040 on pgd_linf attack with eps = 0.01568627450980392


Evaluating: 100%|██████████| 157/157 [00:13<00:00, 11.37it/s]


Standard accuracy 0.78240
Robust accuracy 0.00000 on pgd_linf attack with eps = 0.03137254901960784


Evaluating: 100%|██████████| 157/157 [00:13<00:00, 11.37it/s]

Standard accuracy 0.78240
Robust accuracy 0.00000 on pgd_linf attack with eps = 0.06274509803921569


In [18]:
## FGSM Linf Robust Accuracy
for epsilon in [2/255, 4/255, 8/255, 16/255]:
    test_model_on_single_attack(normal_model, attack='pgd_linf', eps=epsilon, k=1)

Evaluating: 100%|██████████| 157/157 [00:02<00:00, 69.88it/s]


Standard accuracy 0.78240
Robust accuracy 0.60900 on pgd_linf attack with eps = 0.00784313725490196


Evaluating: 100%|██████████| 157/157 [00:02<00:00, 69.38it/s]


Standard accuracy 0.78240
Robust accuracy 0.40300 on pgd_linf attack with eps = 0.01568627450980392


Evaluating: 100%|██████████| 157/157 [00:02<00:00, 69.55it/s]


Standard accuracy 0.78240
Robust accuracy 0.14170 on pgd_linf attack with eps = 0.03137254901960784


Evaluating: 100%|██████████| 157/157 [00:02<00:00, 70.20it/s]

Standard accuracy 0.78240
Robust accuracy 0.01610 on pgd_linf attack with eps = 0.06274509803921569


In [19]:
## L2 Robust Accuracy
for epsilon in [0.25, 0.5, 0.75]:
    test_model_on_single_attack(normal_model, attack='pgd_l2', eps=epsilon)

Evaluating: 100%|██████████| 157/157 [00:13<00:00, 11.40it/s]


Standard accuracy 0.78240
Robust accuracy 0.12660 on pgd_l2 attack with eps = 0.25


Evaluating: 100%|██████████| 157/157 [00:13<00:00, 11.39it/s]


Standard accuracy 0.78240
Robust accuracy 0.00150 on pgd_l2 attack with eps = 0.5


Evaluating: 100%|██████████| 157/157 [00:13<00:00, 11.42it/s]

Standard accuracy 0.78240
Robust accuracy 0.00000 on pgd_l2 attack with eps = 0.75


In [20]:
## FGSM L2 Robust Accuracy
for epsilon in [0.25, 0.5, 0.75]:
    test_model_on_single_attack(normal_model, attack='pgd_l2', eps=epsilon, k=1)

Evaluating: 100%|██████████| 157/157 [00:02<00:00, 69.88it/s]


Standard accuracy 0.78240
Robust accuracy 0.64220 on pgd_l2 attack with eps = 0.25


Evaluating: 100%|██████████| 157/157 [00:02<00:00, 70.69it/s]


Standard accuracy 0.78240
Robust accuracy 0.46490 on pgd_l2 attack with eps = 0.5


Evaluating: 100%|██████████| 157/157 [00:02<00:00, 69.80it/s]

Standard accuracy 0.78240
Robust accuracy 0.31950 on pgd_l2 attack with eps = 0.75
